In [5]:
import numpy as np
import math
import random
from collections import namedtuple
import heapq
import warnings
import time
import json
from datetime import datetime

warnings.filterwarnings('ignore', category=RuntimeWarning)
# ==================== 数据结构定义 ====================
Node = namedtuple('Node', ['x', 'y', 'demand'])

class Zone:
    """禁飞区：任意多边形（凸或凹），用顶点列表表示"""
    def __init__(self, vertices, active_start, active_end):
        """
        vertices: 多边形顶点列表 [(x1,y1), (x2,y2), ...]，按顺时针或逆时针顺序
        active_start, active_end: 禁飞时段
        """
        self.vertices = vertices
        self.active_start = active_start
        self.active_end = active_end
        # 预计算包围盒加速
        self.min_x = min(v[0] for v in vertices)
        self.max_x = max(v[0] for v in vertices)
        self.min_y = min(v[1] for v in vertices)
        self.max_y = max(v[1] for v in vertices)
    
    def contains(self, x, y, time):
        """判断点(x,y)在时刻time是否在多边形禁飞区内"""
        if not (self.active_start <= time <= self.active_end):
            return False
        
        # 快速剔除：包围盒检查
        if not (self.min_x <= x <= self.max_x and self.min_y <= y <= self.max_y):
            return False
        
        # 射线法判断点是否在多边形内
        inside = False
        n = len(self.vertices)
        for i in range(n):
            x1, y1 = self.vertices[i]
            x2, y2 = self.vertices[(i + 1) % n]
            
            # 检查射线是否与边相交
            if ((y1 > y) != (y2 > y)) and (x < (x2 - x1) * (y - y1) / (y2 - y1) + x1):
                inside = not inside
        
        return inside
    
    def line_intersects(self, x1, y1, x2, y2, time):
        """判断线段是否与多边形相交（考虑时间）"""
        if not (self.active_start <= time <= self.active_end):
            return False
        
        # 情况1：端点在内
        if self.contains(x1, y1, time) or self.contains(x2, y2, time):
            return True
        
        # 情况2：线段与任何边相交
        n = len(self.vertices)
        for i in range(n):
            x3, y3 = self.vertices[i]
            x4, y4 = self.vertices[(i + 1) % n]
            
            if self._segments_intersect(x1, y1, x2, y2, x3, y3, x4, y4):
                return True
        
        return False
    
    def _segments_intersect(self, x1, y1, x2, y2, x3, y3, x4, y4):
        """
        判断两条线段是否相交（包括端点接触）
        使用方向法（orientation test）
        """
        def orientation(ax, ay, bx, by, cx, cy):
            """计算三点方向：>0 逆时针，<0 顺时针，=0 共线"""
            return (bx - ax) * (cy - ay) - (by - ay) * (cx - ax)
        
        def on_segment(ax, ay, bx, by, cx, cy):
            """检查点c是否在线段ab上（假设三点共线）"""
            return (min(ax, bx) <= cx <= max(ax, bx) and 
                    min(ay, by) <= cy <= max(ay, by))
        
        o1 = orientation(x1, y1, x2, y2, x3, y3)
        o2 = orientation(x1, y1, x2, y2, x4, y4)
        o3 = orientation(x3, y3, x4, y4, x1, y1)
        o4 = orientation(x3, y3, x4, y4, x2, y2)
        
        # 一般情况：线段相交
        if o1 * o2 < 0 and o3 * o4 < 0:
            return True
        
        # 特殊情况：端点在线段上
        if o1 == 0 and on_segment(x1, y1, x2, y2, x3, y3):
            return True
        if o2 == 0 and on_segment(x1, y1, x2, y2, x4, y4):
            return True
        if o3 == 0 and on_segment(x3, y3, x4, y4, x1, y1):
            return True
        if o4 == 0 and on_segment(x3, y3, x4, y4, x2, y2):
            return True
        
        return False

Problem = namedtuple('Problem', [
    'nodes',            # list of Node, 索引0为配送中心
    'truck_speed',      # 卡车速度
    'drone_speed',      # 无人机速度
    'truck_capacity',   # 卡车最大载重
    'drone_capacity',   # 无人机最大载重
    'truck_max_dist',   # 卡车最大行驶距离
    'drone_max_dist',   # 无人机单次最大飞行距离
    'num_trucks',       # 卡车数量
    'truck_feasible',   # 卡车弧可行性矩阵 (n*n) bool
    'no_fly_zones',     # list of Zone
])

# ==================== 工具函数 ====================
def euclidean_distance(p1, p2):
    return math.hypot(p1.x - p2.x, p1.y - p2.y)

def build_distance_matrix(nodes):
    n = len(nodes)
    dist = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            dist[i, j] = euclidean_distance(nodes[i], nodes[j])
    return dist


# ==================== 动态无人机距离函数 ====================
def build_drone_distance_function(nodes, zones):
    """
    返回一个函数，给定起点索引i、终点索引j、当前时间，返回考虑禁飞区绕行后的飞行距离
    使用A*计算，并加入简单缓存避免重复计算
    支持任意多边形禁飞区（通过Zone类的contains和line_intersects方法）
    """
    cache = {}  # (i, j, time_key) -> distance

    def get_distance(i, j, current_time):
        if i == j:
            return 0.0
        if math.isinf(current_time):
            return float('inf')
        # 时间离散化以减少缓存键爆炸（可根据需要调整）
        time_key = int(current_time)  # 按整秒离散
        key = (i, j, time_key)
        if key in cache:
            return cache[key]

        x1, y1 = nodes[i].x, nodes[i].y
        x2, y2 = nodes[j].x, nodes[j].y

        # 检查直线路径是否与任何禁飞区相交（在当前时间）
        need_detour = False
        for zone in zones:
            if zone.line_intersects(x1, y1, x2, y2, current_time):
                need_detour = True
                break

        if not need_detour:
            # 直线可飞
            dist = math.hypot(x2 - x1, y2 - y1)
            cache[key] = dist
            return dist
        else:
            # 需要绕行，使用A*
            # 获取地图边界（基于所有节点和禁飞区顶点）
            all_x = [node.x for node in nodes]
            all_y = [node.y for node in nodes]
            for zone in zones:
                for vx, vy in zone.vertices:
                    all_x.append(vx)
                    all_y.append(vy)
            
            # 添加一些边界余量
            margin = 20
            min_x = min(all_x) - margin
            max_x = max(all_x) + margin
            min_y = min(all_y) - margin
            max_y = max(all_y) + margin

            grid_res = 2.0  # 网格分辨率，可根据精度调整

            def to_grid(x, y):
                gx = int(round((x - min_x) / grid_res))
                gy = int(round((y - min_y) / grid_res))
                return gx, gy

            start_gx, start_gy = to_grid(x1, y1)
            goal_gx, goal_gy = to_grid(x2, y2)

            # 检查起点/终点是否在禁飞区内（不可飞行）
            if any(zone.contains(x1, y1, current_time) for zone in zones) or \
               any(zone.contains(x2, y2, current_time) for zone in zones):
                cache[key] = float('inf')
                return float('inf')

            # 8邻域移动方向及代价
            directions = [
                (1, 0, grid_res), (-1, 0, grid_res),
                (0, 1, grid_res), (0, -1, grid_res),
                (1, 1, math.sqrt(2)*grid_res), (1, -1, math.sqrt(2)*grid_res),
                (-1, 1, math.sqrt(2)*grid_res), (-1, -1, math.sqrt(2)*grid_res)
            ]

            def heuristic(gx, gy):
                dx = (gx - goal_gx) * grid_res
                dy = (gy - goal_gy) * grid_res
                return math.hypot(dx, dy)

            def is_blocked(gx, gy):
                """检查网格是否被任何激活的禁飞区覆盖"""
                x = min_x + gx * grid_res + grid_res/2
                y = min_y + gy * grid_res + grid_res/2
                return any(zone.contains(x, y, current_time) for zone in zones)

            open_set = []
            heapq.heappush(open_set, (0, start_gx, start_gy))
            g_score = {(start_gx, start_gy): 0}
            f_score = {(start_gx, start_gy): heuristic(start_gx, start_gy)}
            closed_set = set()

            while open_set:
                _, gx, gy = heapq.heappop(open_set)
                if (gx, gy) in closed_set:
                    continue
                
                if (gx, gy) == (goal_gx, goal_gy):
                    dist = g_score[(gx, gy)]
                    cache[key] = dist
                    return dist
                
                closed_set.add((gx, gy))

                for dx, dy, cost in directions:
                    ngx, ngy = gx + dx, gy + dy
                    
                    # 边界检查
                    if ngx < 0 or ngy < 0:
                        continue
                    
                    # 粗略估算网格范围防止无限扩展
                    if ngx * grid_res > (max_x - min_x) or ngy * grid_res > (max_y - min_y):
                        continue
                    
                    if is_blocked(ngx, ngy):
                        continue
                    
                    tentative_g = g_score[(gx, gy)] + cost
                    
                    if (ngx, ngy) not in g_score or tentative_g < g_score[(ngx, ngy)]:
                        g_score[(ngx, ngy)] = tentative_g
                        f = tentative_g + heuristic(ngx, ngy)
                        f_score[(ngx, ngy)] = f
                        heapq.heappush(open_set, (f, ngx, ngy))

            # 无法找到路径
            cache[key] = float('inf')
            return float('inf')

    return get_distance

# ==================== 无人机任务可行性检查 ====================
def check_drone_mission_feasibility(launch_idx, cust_idx, rendezvous_idx, prob, drone_dist_func, current_time):
    """
    检查无人机任务的可行性（完全动态）
    """
    # 客户需求不超过无人机负载
    if prob.nodes[cust_idx].demand > prob.drone_capacity:
        return False

    # 计算第一段飞行距离（已包含禁飞区绕行）
    d1 = drone_dist_func(launch_idx, cust_idx, current_time)
    if d1 == float('inf'):
        return False  # 第一段不可行

    # 计算第二段飞行距离
    d2 = drone_dist_func(cust_idx, rendezvous_idx, current_time + d1 / prob.drone_speed)
    if d2 == float('inf'):
        return False  # 第二段不可行

    if d1 + d2 > prob.drone_max_dist:
        return False

    return True

# ==================== 初始解生成 ====================
def initial_solution(prob, dist):
    """
    顺序插入法生成初始解，确保所有顾客被分配到卡车路径中
    并满足卡车容量和距离约束（无人机任务后续优化）
    """
    n_customers = len(prob.nodes) - 1
    customers = list(range(1, n_customers + 1))
    random.shuffle(customers)
    routes = []
    route_distances = []

    for cust in customers:
        inserted = False
        for r_idx, route in enumerate(routes):
            # 检查容量
            current_demand = sum(prob.nodes[node].demand for node in route if node != 0)
            if current_demand + prob.nodes[cust].demand > prob.truck_capacity:
                continue

            # 尝试所有插入位置
            for pos in range(1, len(route)):
                new_route = route[:pos] + [cust] + route[pos:]
                # 检查卡车弧可行性及行驶距离
                feasible = True
                new_dist = 0
                for i in range(len(new_route) - 1):
                    u, v = new_route[i], new_route[i + 1]
                    if not prob.truck_feasible[u, v]:
                        feasible = False
                        break
                    new_dist += dist[u, v]
                if not feasible:
                    continue
                if new_dist > prob.truck_max_dist:
                    continue

                # 插入可行
                route[:] = new_route
                route_distances[r_idx] = new_dist
                inserted = True
                break
            if inserted:
                break

        if not inserted:
            # 创建新路径
            new_route = [0, cust, 0]
            if prob.truck_feasible[0, cust] and prob.truck_feasible[cust, 0]:
                d = dist[0, cust] + dist[cust, 0]
                if d <= prob.truck_max_dist:
                    routes.append(new_route)
                    route_distances.append(d)
                else:
                    print(f"警告：客户{cust}往返距离超过卡车最大行驶距离，仍添加")
                    routes.append(new_route)
                    route_distances.append(d)
            else:
                print(f"警告：客户{cust}无法通过卡车直接往返，仍添加")
                routes.append(new_route)
                route_distances.append(dist[0, cust] + dist[cust, 0])

    return {i: routes[i] for i in range(len(routes))}

# ==================== 无人机路径优化 ====================
def optimize_drone_route(truck_route, prob, dist, drone_dist_func, current_time=0):
    """
    优化无人机任务分配，完全动态考虑禁飞区
    """
    n = len(truck_route)
    graph = [[] for _ in range(n + 1)]

    # 实线弧（卡车直接行驶）
    for i in range(n - 1):
        u_node = truck_route[i]
        v_node = truck_route[i + 1]
        if prob.truck_feasible[u_node, v_node]:
            truck_time = dist[u_node, v_node] / prob.truck_speed
            graph[i].append((i + 1, truck_time, None))

    # 虚线弧（无人机任务）
    for i in range(n - 2):
        u_node = truck_route[i]
        j = i + 2
        v_node = truck_route[j]
        k_node = truck_route[i + 1]

        # 估计到达起飞点的时间（忽略等待，用于可行性预检查）
        mission_time_est = current_time
        for idx in range(i):
            mission_time_est += dist[truck_route[idx], truck_route[idx+1]] / prob.truck_speed

        if not check_drone_mission_feasibility(u_node, k_node, v_node, prob, drone_dist_func, mission_time_est):
            continue

        d1 = drone_dist_func(u_node, k_node, mission_time_est)
        d2 = drone_dist_func(k_node, v_node, mission_time_est + d1 / prob.drone_speed)
        drone_time = (d1 + d2) / prob.drone_speed
        truck_time = dist[u_node, v_node] / prob.truck_speed
        weight = max(truck_time, drone_time)
        # 存储起飞点位置索引 i，用于后续获取准确起飞时间
        edge_info = (i, u_node, k_node, v_node, drone_time, truck_time)
        graph[i].append((j, weight, edge_info))

    # 虚拟终点
    graph[n-1].append((n, 0, None))

    # Dijkstra 求最短路径（按完成时间）
    start = 0
    dist_arr = [math.inf] * (n + 1)
    dist_arr[start] = current_time
    prev_node = [None] * (n + 1)
    prev_edge = [None] * (n + 1)
    pq = [(current_time, start)]

    while pq:
        d, u = heapq.heappop(pq)
        if d > dist_arr[u]:
            continue
        for v, w, info in graph[u]:
            nd = d + w
            if nd < dist_arr[v]:
                dist_arr[v] = nd
                prev_node[v] = u
                prev_edge[v] = info
                heapq.heappush(pq, (nd, v))

    # 回溯路径
    path_indices = []
    cur = n
    while cur is not None:
        path_indices.append(cur)
        cur = prev_node[cur]
    path_indices.reverse()

    truck_actual = [truck_route[idx] for idx in path_indices if idx < n]
    drone_assignments = []          # 每个元素为 (launch, cust, rendezvous)
    drone_takeoff_times = []        # 对应每个任务的起飞时间
    wait_drone_total = 0.0
    wait_truck_total = 0.0

    for i in range(len(path_indices) - 1):
        v_idx = path_indices[i + 1]
        info = prev_edge[v_idx]
        if info is not None:
            pos_i, u_node, k_node, v_node, drone_time, truck_time = info
            drone_assignments.append((u_node, k_node, v_node))
            drone_takeoff_times.append(dist_arr[pos_i])   # 准确起飞时间
            if drone_time > truck_time:
                wait_drone_total += drone_time - truck_time
            else:
                wait_truck_total += truck_time - drone_time

    completion_time = dist_arr[n]

    # 卡车行驶距离
    truck_dist = 0.0
    for i in range(len(truck_actual) - 1):
        truck_dist += dist[truck_actual[i]][truck_actual[i + 1]]

    # 无人机飞行距离（使用准确起飞时间）
    drone_dist = 0.0
    for idx, (launch, cust, rendezvous) in enumerate(drone_assignments):
        takeoff_time = drone_takeoff_times[idx]
        d1 = drone_dist_func(launch, cust, takeoff_time)
        d2 = drone_dist_func(cust, rendezvous, takeoff_time + d1 / prob.drone_speed)
        drone_dist += d1 + d2

    return truck_actual, drone_assignments, completion_time, truck_dist, drone_dist, wait_drone_total, wait_truck_total

# ==================== 邻域算子 ====================
def _is_route_feasible(route, prob, dist):
    """检查路径可行性（卡车容量和距离）"""
    demand = sum(prob.nodes[node].demand for node in route if node != 0)
    if demand > prob.truck_capacity:
        return False
    total_dist = 0
    for i in range(len(route) - 1):
        u, v = route[i], route[i + 1]
        if not prob.truck_feasible[u, v]:
            return False
        total_dist += dist[u, v]
    if total_dist > prob.truck_max_dist:
        return False
    return True

def operator_insert(solution, prob, dist, drone_dist_func):
    truck_ids = list(solution.keys())
    if len(truck_ids) < 2:
        return None
    src_id = random.choice(truck_ids)
    tgt_id = random.choice([tid for tid in truck_ids if tid != src_id])
    src_route = solution[src_id][:]
    tgt_route = solution[tgt_id][:]
    src_inner = src_route[1:-1]
    if not src_inner:
        return None
    l = random.randint(0, len(src_inner) - 1)
    r = random.randint(l, len(src_inner) - 1)
    segment = src_inner[l:r + 1]
    new_src = [src_route[0]] + src_inner[:l] + src_inner[r + 1:] + [src_route[-1]]
    if not _is_route_feasible(new_src, prob, dist):
        return None
    insert_positions = list(range(len(tgt_route) - 1))
    random.shuffle(insert_positions)
    for pos in insert_positions:
        new_tgt = tgt_route[:pos + 1] + segment + tgt_route[pos + 1:]
        if _is_route_feasible(new_tgt, prob, dist):
            new_solution = solution.copy()
            new_solution[src_id] = new_src
            new_solution[tgt_id] = new_tgt
            info = ('insert', src_id, tgt_id, tuple(segment))
            return new_solution, info
    return None

def operator_swap(solution, prob, dist, drone_dist_func):
    truck_ids = list(solution.keys())
    if not truck_ids:
        return None
    src_id = random.choice(truck_ids)
    route = solution[src_id]
    inner_indices = list(range(1, len(route) - 1))
    if len(inner_indices) < 2:
        return None
    i, j = random.sample(inner_indices, 2)
    new_route = route[:]
    new_route[i], new_route[j] = new_route[j], new_route[i]
    if not _is_route_feasible(new_route, prob, dist):
        return None
    new_solution = solution.copy()
    new_solution[src_id] = new_route
    info = ('swap', src_id, tuple(sorted([i, j])))
    return new_solution, info

def operator_2opt(solution, prob, dist, drone_dist_func):
    truck_ids = list(solution.keys())
    if not truck_ids:
        return None
    src_id = random.choice(truck_ids)
    route = solution[src_id]
    inner_indices = list(range(1, len(route) - 1))
    if len(inner_indices) < 2:
        return None
    l = random.choice(inner_indices)
    r = random.choice(inner_indices)
    if l > r:
        l, r = r, l
    new_route = route[:l] + route[l:r+1][::-1] + route[r+1:]
    if not _is_route_feasible(new_route, prob, dist):
        return None
    new_solution = solution.copy()
    new_solution[src_id] = new_route
    info = ('2opt', src_id, tuple(sorted([l, r])))
    return new_solution, info

def operator_relocate(solution, prob, dist, drone_dist_func):
    """
    将一个客户从一条路径移动到另一条路径
    """
    truck_ids = list(solution.keys())
    if len(truck_ids) < 2:
        return None
    
    src_id = random.choice(truck_ids)
    tgt_id = random.choice([tid for tid in truck_ids if tid != src_id])
    
    src_route = solution[src_id][:]
    tgt_route = solution[tgt_id][:]
    
    # 选择要移动的客户
    movable = [i for i in range(1, len(src_route)-1)]
    if not movable:
        return None
    
    cust_idx = random.choice(movable)
    customer = src_route[cust_idx]
    
    # 从源路径移除
    new_src = src_route[:cust_idx] + src_route[cust_idx+1:]
    if not _is_route_feasible(new_src, prob, dist):
        return None
    
    # 尝试插入目标路径
    for pos in range(1, len(tgt_route)):
        new_tgt = tgt_route[:pos] + [customer] + tgt_route[pos:]
        if _is_route_feasible(new_tgt, prob, dist):
            new_solution = solution.copy()
            new_solution[src_id] = new_src
            new_solution[tgt_id] = new_tgt
            return new_solution, ('relocate', src_id, tgt_id, customer)
    
    return None

def operator_exchange(solution, prob, dist, drone_dist_func):
    """
    交换两条路径中的客户
    """
    truck_ids = list(solution.keys())
    if len(truck_ids) < 2:
        return None
    
    id1, id2 = random.sample(truck_ids, 2)
    route1 = solution[id1][:]
    route2 = solution[id2][:]
    
    # 选择要交换的客户
    idx1_options = [i for i in range(1, len(route1)-1)]
    idx2_options = [i for i in range(1, len(route2)-1)]
    
    if not idx1_options or not idx2_options:
        return None
    
    idx1 = random.choice(idx1_options)
    idx2 = random.choice(idx2_options)
    
    cust1 = route1[idx1]
    cust2 = route2[idx2]
    
    # 交换客户
    new_route1 = route1[:idx1] + [cust2] + route1[idx1+1:]
    new_route2 = route2[:idx2] + [cust1] + route2[idx2+1:]
    
    if _is_route_feasible(new_route1, prob, dist) and _is_route_feasible(new_route2, prob, dist):
        new_solution = solution.copy()
        new_solution[id1] = new_route1
        new_solution[id2] = new_route2
        return new_solution, ('exchange', id1, id2, cust1, cust2)
    
    return None

def operator_cross(solution, prob, dist, drone_dist_func):
    """
    交叉交换：交换两条路径的后半部分
    """
    truck_ids = list(solution.keys())
    if len(truck_ids) < 2:
        return None
    
    id1, id2 = random.sample(truck_ids, 2)
    route1 = solution[id1][:]
    route2 = solution[id2][:]
    
    if len(route1) < 3 or len(route2) < 3:
        return None
    
    # 随机选择交叉点
    cross1 = random.randint(1, len(route1)-2)
    cross2 = random.randint(1, len(route2)-2)
    
    # 交换后半部分
    new_route1 = route1[:cross1] + route2[cross2:]
    new_route2 = route2[:cross2] + route1[cross1:]
    
    if _is_route_feasible(new_route1, prob, dist) and _is_route_feasible(new_route2, prob, dist):
        new_solution = solution.copy()
        new_solution[id1] = new_route1
        new_solution[id2] = new_route2
        return new_solution, ('cross', id1, id2, cross1, cross2)
    
    return None

# ==================== 禁忌搜索主体 ====================
class ImprovedSPTS:
    def __init__(self, prob, dist, drone_dist_func, max_iter=5000, tabu_len=15, patience=200,
                 c_truck_km=1.0, c_drone_km=0.1, c_truck_fixed=200, c_drone_fixed=10,
                 c_wait_drone=1.0, c_wait_truck=2.0,
                 temperature=2000, cooling_rate=0.98):
        
        self.prob = prob
        self.dist = dist
        self.drone_dist_func = drone_dist_func
        self.max_iter = max_iter
        self.tabu_len = tabu_len
        self.patience = patience
        self.tabu_list = []
        self.best_solution = None
        self.best_cost = float('inf')
        self.iter_count = 0

        # 成本系数
        self.c_truck_km = c_truck_km
        self.c_drone_km = c_drone_km
        self.c_truck_fixed = c_truck_fixed
        self.c_drone_fixed = c_drone_fixed
        self.c_wait_drone = c_wait_drone
        self.c_wait_truck = c_wait_truck
        
        # 模拟退火参数
        self.temperature = temperature
        self.cooling_rate = cooling_rate
        
        # 算子权重（自适应）
        self.operator_weights = {
            'relocate': 1.0,
            'exchange': 1.0,
            'cross': 1.0,
            'insert': 0.5,
            'swap': 0.5,
            '2opt': 0.5
        }
        self.operator_success = {op: 0 for op in self.operator_weights}
        self.operator_attempts = {op: 0 for op in self.operator_weights}

    def evaluate(self, solution):
        """评估解的成本"""
        total_truck_dist = 0.0
        total_drone_dist = 0.0
        total_wait_drone = 0.0
        total_wait_truck = 0.0
        total_drone_missions = 0

        for route in solution.values():
            if len(route) < 2:
                continue
            _, drone_assignments, _, truck_dist, drone_dist, wait_drone, wait_truck = optimize_drone_route(
                route, self.prob, self.dist, self.drone_dist_func)
            total_truck_dist += truck_dist
            total_drone_dist += drone_dist
            total_wait_drone += wait_drone
            total_wait_truck += wait_truck
            total_drone_missions += len(drone_assignments)

        num_trucks = len(solution)

        cost = (self.c_truck_km * total_truck_dist +
                self.c_drone_km * total_drone_dist +
                self.c_truck_fixed * num_trucks +
                self.c_drone_fixed * total_drone_missions +
                self.c_wait_drone * total_wait_drone +
                self.c_wait_truck * total_wait_truck)
        return cost

    def select_operator(self):
        """根据权重选择算子"""
        total = sum(self.operator_weights.values())
        r = random.random() * total
        cumsum = 0
        for op, weight in self.operator_weights.items():
            cumsum += weight
            if r <= cumsum:
                return op
        return random.choice(list(self.operator_weights.keys()))

    def update_weights(self):
        """更新算子权重"""
        for op in self.operator_weights:
            if self.operator_attempts[op] > 0:
                success_rate = self.operator_success[op] / self.operator_attempts[op]
                self.operator_weights[op] = max(0.1, min(3.0, 0.5 + 3 * success_rate))
        
        # 重置计数器
        self.operator_success = {op: 0 for op in self.operator_success}
        self.operator_attempts = {op: 0 for op in self.operator_attempts}

    def apply_operator(self, op_name, current):
        """应用选中的算子"""
        operators = {
            'relocate': operator_relocate,
            'exchange': operator_exchange,
            'cross': operator_cross,
            'insert': operator_insert,
            'swap': operator_swap,
            '2opt': operator_2opt
        }
        
        if op_name in operators:
            return operators[op_name](current, self.prob, self.dist, self.drone_dist_func)
        return None

    def search(self):
        """主搜索过程"""
        # 使用基础初始解
        current = initial_solution(self.prob, self.dist)
        current_cost = self.evaluate(current)
        self.best_solution = current
        self.best_cost = current_cost

        print("\n=== 基础初始解===")
        for tid, route in current.items():
            print(f"卡车{tid+1}: {route}")
        print(f"初始总成本: {current_cost:.2f}\n")

        no_improve = 0

        for self.iter_count in range(self.max_iter):
            # 每50次迭代更新权重
            if self.iter_count > 0 and self.iter_count % 20 == 0:
                self.update_weights()

            # 生成多个候选解
            candidates = []
            for _ in range(10):  # 每次生成10个候选
                op_name = self.select_operator()
                self.operator_attempts[op_name] += 1

                res = self.apply_operator(op_name, current)
                if res:
                    new_sol, info = res
                    cost = self.evaluate(new_sol)
                    candidates.append((cost, new_sol, info, op_name))

                    if cost < current_cost:
                        self.operator_success[op_name] += 1

            if not candidates:
                continue

            # 排序候选
            candidates.sort(key=lambda x: x[0])

            # 选择最佳非禁忌解
            selected = None
            for cost, sol, info, op_name in candidates:
                if not self.is_tabu(info):
                    # 模拟退火接受准则
                    if cost < current_cost:
                        selected = (cost, sol, info)
                        break
                    else:
                        # 以一定概率接受劣化解（不打印）
                        delta = cost - current_cost
                        accept_prob = math.exp(-delta / max(self.temperature, 0.1))
                        if random.random() < accept_prob:
                            selected = (cost, sol, info)
                            break

            if selected is None and candidates:
                # 如果没有可接受的，选择最好的（打破禁忌）
                best_candidate = candidates[0]
                selected = (best_candidate[0], best_candidate[1], best_candidate[2])

            if selected is None:
                continue

            cost, sol, info = selected

            # 更新禁忌表
            self.add_tabu(info)

            # 移动到新解
            current = sol
            current_cost = cost

            # 更新最优解
            if cost < self.best_cost:
                self.best_solution = sol
                self.best_cost = cost
                no_improve = 0
            else:
                no_improve += 1

            # 降温
            self.temperature *= self.cooling_rate

            # 早停
            if no_improve >= self.patience:
                break

        return self.best_solution, self.best_cost

    def is_tabu(self, op_info):
        """检查是否在禁忌表中"""
        if op_info is None:
            return False
        
        obj = self._extract_tabu_object(op_info)
        if obj is None:
            return False
            
        for tabu_obj, expire in self.tabu_list:
            if tabu_obj == obj and expire > self.iter_count:
                return True
        return False

    def add_tabu(self, op_info):
        """添加禁忌"""
        obj = self._extract_tabu_object(op_info)
        if obj is None:
            return
            
        expire = self.iter_count + self.tabu_len
        self.tabu_list.append((obj, expire))
        # 清理过期禁忌
        self.tabu_list = [(o, e) for (o, e) in self.tabu_list if e > self.iter_count]

    def _extract_tabu_object(self, op_info):
        """从操作信息中提取禁忌对象"""
        if op_info is None:
            return None
            
        op_type = op_info[0]
        
        if op_type == 'relocate':
            _, src_id, tgt_id, customer = op_info
            return ('relocate', src_id, tgt_id, customer)
        elif op_type == 'exchange':
            _, id1, id2, cust1, cust2 = op_info
            return ('exchange', id1, id2, cust1, cust2)
        elif op_type == 'cross':
            _, id1, id2, cross1, cross2 = op_info
            return ('cross', id1, id2)
        elif op_type == 'insert':
            _, src_id, tgt_id, segment = op_info
            return ('insert', src_id, tgt_id, tuple(segment))
        elif op_type == 'swap':
            _, route_id, nodes = op_info
            return ('swap', route_id, nodes)
        elif op_type == '2opt':
            _, route_id, nodes = op_info
            return ('2opt', route_id, nodes)
        else:
            return None
        
# ==================== 主程序示例 ====================
if __name__ == '__main__':
    import pandas as pd
    import os
    import time
    import json
    from datetime import datetime

    # 记录开始时间
    start_time = time.time()

    # 读取Excel文件
    file_path = r"C:\Users\yu\Desktop\Set1_1.xlsx"
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"文件不存在: {file_path}")
    df = pd.read_excel(file_path)

    # 假设列顺序为 x坐标, y坐标, 编号, 需求量
    x_col = df.columns[0]
    y_col = df.columns[1]
    id_col = df.columns[2]
    demand_col = df.columns[3]

    nodes_data = []
    for idx, row in df.iterrows():
        node_id = int(row[id_col])
        x = float(row[x_col])
        y = float(row[y_col])
        demand = float(row[demand_col]) if demand_col in df.columns else 0
        nodes_data.append((x, y, demand, node_id))

    nodes_data.sort(key=lambda x: x[3])
    nodes = [Node(x, y, demand) for x, y, demand, _ in nodes_data]
    n = len(nodes)
    print(f"读取了{n}个节点，其中配送中心: {nodes[0].x},{nodes[0].y}，客户数: {n-1}")

    # 定义三个不重叠的禁飞区
    # 示例：三角形禁飞区
    zone1 = Zone([(40, 40), (60, 40), (50, 60)], active_start=0, active_end=24)

    # 示例：五边形禁飞区
    zone2 = Zone([(-70, -20), (-85, -25), (-90, -40), (-75, -45), (-65, -30)], 
                 active_start=8, active_end=20)

    # 示例：四边形禁飞区
    zone3 = Zone([(-10, 60), (-30, 65), (-25, 80), (-5, 75)], 
                 active_start=12, active_end=16)

    no_fly_zones = [zone1, zone2, zone3]


    # 构建距离矩阵
    dist = build_distance_matrix(nodes)

    # 构建动态无人机距离函数
    drone_dist_func = build_drone_distance_function(nodes, no_fly_zones)

    # 构建卡车可行性矩阵（假设所有点之间可行，除自环）
    truck_feasible = np.ones((n, n), dtype=bool)
    for i in range(n):
        truck_feasible[i, i] = False

    # 设置问题参数
    total_demand = sum(node.demand for node in nodes[1:])
    estimated_trucks = math.ceil(total_demand / 50)

    prob = Problem(
        nodes=nodes,
        truck_speed=50,
        drone_speed=75,
        truck_capacity=1000,
        drone_capacity=3,
        truck_max_dist=500,
        drone_max_dist=30,
        num_trucks=estimated_trucks,
        truck_feasible=truck_feasible,
        no_fly_zones=no_fly_zones
    )
    
    # 记录求解开始时间
    solve_start_time = time.time()
    
    # 创建改进的求解器
    solver = ImprovedSPTS(
        prob, dist, drone_dist_func,
        max_iter=5000,
        tabu_len=15,
        patience=200,
        c_truck_km=1.0,
        c_drone_km=0.1,
        c_truck_fixed=200,
        c_drone_fixed=10,
        c_wait_drone=1.0,
        c_wait_truck=2.0,
        temperature=2000,
        cooling_rate=0.98
    )

    best_sol, best_cost = solver.search()
    
    # 记录求解结束时间
    solve_end_time = time.time()
    total_solve_time = solve_end_time - solve_start_time
    
    print(f"\n求解时间: {total_solve_time:.2f} 秒")
    print(f"\n最优总成本: {best_cost:.2f}")
    print(f"使用的卡车数: {len(best_sol)}")
    print("\n详细路径方案:")

    total_truck_dist = 0
    total_drone_dist = 0
    total_missions = 0

    for tid, route in best_sol.items():
        print(f"\n卡车{tid+1} 原始路径: {route}")
        truck_actual, drone_assignments, comp_time, truck_dist, drone_dist, wait_drone, wait_truck = optimize_drone_route(
            route, prob, dist, drone_dist_func)
        print(f"  卡车实际访问: {truck_actual}")
        print(f"  卡车行驶距离: {truck_dist:.2f}")
        print(f"  无人机飞行距离: {drone_dist:.2f}")
        if drone_assignments:
            print(f"  无人机任务数: {len(drone_assignments)}")
            for i, (launch, cust, rendezvous) in enumerate(drone_assignments):
                print(f"    任务{i+1}: 从{launch}起飞 → 服务顾客{cust} → 在{rendezvous}会合")
        else:
            print(f"  无无人机任务")
        print(f"  等待时间 - 无人机等卡车: {wait_drone:.2f}, 卡车等无人机: {wait_truck:.2f}")
        print(f"  路径完成时间: {comp_time:.2f}")

        total_truck_dist += truck_dist
        total_drone_dist += drone_dist
        total_missions += len(drone_assignments)

    print(f"\n=== 汇总 ===")
    print(f"总卡车行驶距离: {total_truck_dist:.2f}")
    print(f"总无人机飞行距离: {total_drone_dist:.2f}")
    print(f"总无人机任务数: {total_missions}")
    
    # 约束检查
    print(f"\n=== 约束检查 ===")
    served_customers = set()
    for route in best_sol.values():
        served_customers.update(route[1:-1])
    all_customers = set(range(1, n))
    if served_customers == all_customers:
        print("所有客户都被服务")
    else:
        print(f"未服务的客户: {all_customers - served_customers}")

    capacity_violations = 0
    distance_violations = 0
    for route in best_sol.values():
        demand = sum(prob.nodes[node].demand for node in route if node != 0)
        if demand > prob.truck_capacity:
            print(f"路径 {route} 容量超限: {demand} > {prob.truck_capacity}")
            capacity_violations += 1
        route_dist = 0
        for i in range(len(route)-1):
            route_dist += dist[route[i], route[i+1]]
        if route_dist > prob.truck_max_dist:
            print(f"路径 {route} 距离超限: {route_dist:.2f} > {prob.truck_max_dist}")
            distance_violations += 1
    if capacity_violations == 0:
        print("所有路径满足容量约束")
    if distance_violations == 0:
        print("所有路径满足距离约束")

    drone_violations = 0
    for tid, route in best_sol.items():
        _, drone_assignments, _, _, _, _, _ = optimize_drone_route(route, prob, dist, drone_dist_func)
        for launch, cust, rendezvous in drone_assignments:
            if prob.nodes[cust].demand > prob.drone_capacity:
                print(f"无人机任务 {launch}->{cust}->{rendezvous} 需求超限: {prob.nodes[cust].demand} > {prob.drone_capacity}")
                drone_violations += 1
            # 动态可行性已在优化时保证，这里不再重复检查
    if drone_violations == 0:
        print("所有无人机任务满足约束")

    # 创建Excel写入器
    output_file = r"C:\Users\yu\Desktop\Set1_1.xlsx"
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        
        # ==================== Sheet 1: 问题信息 ====================
        problem_info_df = pd.DataFrame([
            ['求解时间', datetime.now().strftime('%Y-%m-%d %H:%M:%S')],
            ['求解耗时(秒)', f"{total_solve_time:.2f}"],
            ['文件路径', file_path],
            ['节点总数', n],
            ['客户数量', n-1],
            ['总需求', f"{total_demand:.1f}"],
            ['估计卡车数', estimated_trucks],
            ['实际使用卡车数', len(best_sol)],
            ['卡车速度', prob.truck_speed],
            ['无人机速度', prob.drone_speed],
            ['卡车容量', prob.truck_capacity],
            ['无人机容量', prob.drone_capacity],
            ['卡车最大距离', prob.truck_max_dist],
            ['无人机最大距离', prob.drone_max_dist],
            ['总成本', f"{best_cost:.2f}"],
            ['总卡车行驶距离', f"{total_truck_dist:.2f}"],
            ['总无人机飞行距离', f"{total_drone_dist:.2f}"],
            ['总无人机任务数', total_missions]
        ], columns=['参数', '值'])
        problem_info_df.to_excel(writer, sheet_name='问题信息', index=False)
        
        # ==================== Sheet 2: 路径详情 ====================
        routes_data = []
        for tid, route in best_sol.items():
            truck_actual, drone_assignments, comp_time, truck_dist, drone_dist, wait_drone, wait_truck = optimize_drone_route(
                route, prob, dist, drone_dist_func)
            
            routes_data.append({
                '卡车编号': tid+1,
                '原始路径': ' -> '.join(map(str, route)),
                '实际路径': ' -> '.join(map(str, truck_actual)),
                '卡车行驶距离': f"{truck_dist:.2f}",
                '无人机飞行距离': f"{drone_dist:.2f}",
                '无人机任务数': len(drone_assignments),
                '完成时间': f"{comp_time:.2f}",
                '无人机等待时间': f"{wait_drone:.2f}",
                '卡车等待时间': f"{wait_truck:.2f}"
            })
        
        routes_df = pd.DataFrame(routes_data)
        routes_df.to_excel(writer, sheet_name='路径详情', index=False)
        
        # ==================== Sheet 3: 无人机任务详情 ====================
        drone_missions_data = []
        for tid, route in best_sol.items():
            truck_actual, drone_assignments, comp_time, truck_dist, drone_dist, wait_drone, wait_truck = optimize_drone_route(
                route, prob, dist, drone_dist_func)
            
            for i, (launch, cust, rendezvous) in enumerate(drone_assignments):
                # 计算无人机飞行距离
                d1 = drone_dist_func(launch, cust, 0)
                d2 = drone_dist_func(cust, rendezvous, d1 / prob.drone_speed)
                
                drone_missions_data.append({
                    '卡车编号': tid+1,
                    '任务编号': i+1,
                    '起飞点': launch,
                    '服务客户': cust,
                    '会合点': rendezvous,
                    '第一段距离': f"{d1:.2f}" if not math.isinf(d1) else "不可行",
                    '第二段距离': f"{d2:.2f}" if not math.isinf(d2) else "不可行",
                    '总飞行距离': f"{d1+d2:.2f}" if not math.isinf(d1+d2) else "不可行",
                    '客户需求': prob.nodes[cust].demand
                })
        
        if drone_missions_data:
            drone_df = pd.DataFrame(drone_missions_data)
            drone_df.to_excel(writer, sheet_name='无人机任务详情', index=False)
        else:
            # 如果没有无人机任务，创建空表
            pd.DataFrame(['无无人机任务'], columns=['信息']).to_excel(writer, sheet_name='无人机任务详情', index=False)
        
        # ==================== Sheet 4: 约束检查结果 ====================
        # 检查所有客户是否被服务
        served_customers = set()
        for route in best_sol.values():
            served_customers.update(route[1:-1])
        for tid, route in best_sol.items():
            _, drone_assignments, _, _, _, _, _ = optimize_drone_route(route, prob, dist, drone_dist_func)
            for launch, cust, rendezvous in drone_assignments:
                served_customers.add(cust)
        
        all_customers = set(range(1, n))
        unserved = all_customers - served_customers
        
        constraint_data = []
        
        # 客户服务检查
        constraint_data.append({
            '检查项': '客户服务完整性',
            '状态': '通过' if len(unserved) == 0 else '失败',
            '详细信息': f"已服务{len(served_customers)}/{n-1}个客户" if len(unserved) == 0 else f"未服务客户: {sorted(unserved)}"
        })
        
        # 卡车容量检查
        capacity_violations = []
        for route in best_sol.values():
            demand = sum(prob.nodes[node].demand for node in route if node != 0)
            if demand > prob.truck_capacity:
                capacity_violations.append({
                    'route': route,
                    'demand': demand,
                    'capacity': prob.truck_capacity
                })
        
        constraint_data.append({
            '检查项': '卡车容量约束',
            '状态': '通过' if len(capacity_violations) == 0 else '失败',
            '详细信息': f"发现{len(capacity_violations)}条路径容量超限" if capacity_violations else "所有路径满足容量约束"
        })
        
        # 卡车距离检查
        distance_violations = []
        for route in best_sol.values():
            route_dist = 0
            for i in range(len(route)-1):
                route_dist += dist[route[i], route[i+1]]
            if route_dist > prob.truck_max_dist:
                distance_violations.append({
                    'route': route,
                    'distance': route_dist,
                    'max_distance': prob.truck_max_dist
                })
        
        constraint_data.append({
            '检查项': '卡车距离约束',
            '状态': '通过' if len(distance_violations) == 0 else '失败',
            '详细信息': f"发现{len(distance_violations)}条路径距离超限" if distance_violations else "所有路径满足距离约束"
        })
        
        # 无人机容量检查
        drone_cap_violations = []
        for tid, route in best_sol.items():
            _, drone_assignments, _, _, _, _, _ = optimize_drone_route(route, prob, dist, drone_dist_func)
            for launch, cust, rendezvous in drone_assignments:
                if prob.nodes[cust].demand > prob.drone_capacity:
                    drone_cap_violations.append({
                        'mission': (launch, cust, rendezvous),
                        'demand': prob.nodes[cust].demand,
                        'capacity': prob.drone_capacity
                    })
        
        constraint_data.append({
            '检查项': '无人机容量约束',
            '状态': '通过' if len(drone_cap_violations) == 0 else '失败',
            '详细信息': f"发现{len(drone_cap_violations)}个无人机任务容量超限" if drone_cap_violations else "所有无人机任务满足容量约束"
        })
        
        # 无人机距离检查
        drone_dist_violations = []
        for tid, route in best_sol.items():
            _, drone_assignments, _, _, _, _, _ = optimize_drone_route(route, prob, dist, drone_dist_func)
            for launch, cust, rendezvous in drone_assignments:
                d1 = drone_dist_func(launch, cust, 0)
                d2 = drone_dist_func(cust, rendezvous, d1 / prob.drone_speed)
                total_dist = d1 + d2
                if total_dist > prob.drone_max_dist:
                    drone_dist_violations.append({
                        'mission': (launch, cust, rendezvous),
                        'distance': total_dist,
                        'max_distance': prob.drone_max_dist
                    })
        
        constraint_data.append({
            '检查项': '无人机距离约束',
            '状态': '通过' if len(drone_dist_violations) == 0 else '失败',
            '详细信息': f"发现{len(drone_dist_violations)}个无人机任务距离超限" if drone_dist_violations else "所有无人机任务满足距离约束"
        })
        
        constraint_df = pd.DataFrame(constraint_data)
        constraint_df.to_excel(writer, sheet_name='约束检查', index=False)
        
        # ==================== Sheet 5: 违规详情（如果有） ====================
        violations_data = []
        
        # 添加容量违规详情
        for violation in capacity_violations:
            violations_data.append({
                '违规类型': '卡车容量超限',
                '路径': ' -> '.join(map(str, violation['route'])),
                '实际值': f"{violation['demand']:.1f}",
                '限制值': violation['capacity'],
                '超出量': f"{violation['demand'] - violation['capacity']:.1f}"
            })
        
        # 添加距离违规详情
        for violation in distance_violations:
            violations_data.append({
                '违规类型': '卡车距离超限',
                '路径': ' -> '.join(map(str, violation['route'])),
                '实际值': f"{violation['distance']:.2f}",
                '限制值': violation['max_distance'],
                '超出量': f"{violation['distance'] - violation['max_distance']:.2f}"
            })
        
        # 添加无人机容量违规详情
        for violation in drone_cap_violations:
            violations_data.append({
                '违规类型': '无人机容量超限',
                '任务': f"{violation['mission'][0]}→{violation['mission'][1]}→{violation['mission'][2]}",
                '实际值': f"{violation['demand']:.1f}",
                '限制值': violation['capacity'],
                '超出量': f"{violation['demand'] - violation['capacity']:.1f}"
            })
        
        # 添加无人机距离违规详情
        for violation in drone_dist_violations:
            violations_data.append({
                '违规类型': '无人机距离超限',
                '任务': f"{violation['mission'][0]}→{violation['mission'][1]}→{violation['mission'][2]}",
                '实际值': f"{violation['distance']:.2f}",
                '限制值': violation['max_distance'],
                '超出量': f"{violation['distance'] - violation['max_distance']:.2f}"
            })
        
        if violations_data:
            violations_df = pd.DataFrame(violations_data)
            violations_df.to_excel(writer, sheet_name='违规详情', index=False)
        
        # ==================== Sheet 6: 未服务客户（如果有） ====================
        if unserved:
            unserved_data = []
            for cust in sorted(unserved):
                unserved_data.append({
                    '客户编号': cust,
                    'x坐标': prob.nodes[cust].x,
                    'y坐标': prob.nodes[cust].y,
                    '需求量': prob.nodes[cust].demand
                })
            unserved_df = pd.DataFrame(unserved_data)
            unserved_df.to_excel(writer, sheet_name='未服务客户', index=False)
    
    print(f"\n结果已保存到Excel文件: {output_file}")
    
    # 在控制台输出求解时间
    print(f"\n总求解时间: {total_solve_time:.2f} 秒")

读取了10个节点，其中配送中心: -37.027966624865954,-50.69654211733612，客户数: 9

=== 基础初始解===
卡车1: [0, 1, 7, 6, 2, 8, 5, 3, 0]
卡车2: [0, 4, 9, 0]
初始总成本: 1164.86


求解时间: 0.69 秒

最优总成本: 875.66
使用的卡车数: 2

详细路径方案:

卡车1 原始路径: [0, 9, 8, 1, 7, 5, 4, 6, 2, 0]
  卡车实际访问: [0, 9, 8, 1, 7, 5, 4, 6, 2, 0]
  卡车行驶距离: 432.72
  无人机飞行距离: 0.00
  无无人机任务
  等待时间 - 无人机等卡车: 0.00, 卡车等无人机: 0.00
  路径完成时间: 8.65

卡车2 原始路径: [0, 3, 0]
  卡车实际访问: [0, 3, 0]
  卡车行驶距离: 42.94
  无人机飞行距离: 0.00
  无无人机任务
  等待时间 - 无人机等卡车: 0.00, 卡车等无人机: 0.00
  路径完成时间: 0.86

=== 汇总 ===
总卡车行驶距离: 475.66
总无人机飞行距离: 0.00
总无人机任务数: 0

=== 约束检查 ===
所有客户都被服务
所有路径满足容量约束
所有路径满足距离约束
所有无人机任务满足约束

结果已保存到Excel文件: C:\Users\yu\Desktop\Set1_1.xlsx

总求解时间: 0.69 秒
